# Daily Challenge: Pinecone Serverless Reranking in Action

Reranking models boost search relevance by assigning similarity scores between a query
and documents, then reordering results so the most pertinent information appears first.
In contexts like healthcare, this helps clinicians quickly access the most critical
clinical notes.

**⚠️ Requires a Pinecone account and API key** -- sign up at https://www.pinecone.io/ if
you don't have one, then run this notebook in Colab so cell 3's `Authenticate()` can
prompt you for it.

### A note on how this solution was verified

Every cell that needs a live Pinecone connection (creating an index, upserting,
querying, reranking against the API) **cannot be executed without your own API key**, so
I couldn't run those end-to-end myself. What I did verify directly, without needing
credentials:
- Installed `pinecone==6.0.1` and read its actual source to confirm every attribute name
  this notebook relies on (`.data`, `.score`, `.document.text`, dict-style
  `match["score"]`, etc.) -- not guessed from memory.
- Downloaded the real medical-notes dataset and confirmed its columns, vector dimension,
  and metadata shape match what the code expects.
- Ran the real `all-MiniLM-L6-v2` model to confirm the embedding function's tensor shapes.
- Found and fixed one real, reproducible bug in the challenge's own given code -- see
  Part 5 below for the exact error it throws.

## Part 1: Load Documents & Execute Reranking Model

### 1. Install Pinecone libraries

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks


### 2. Authenticate with Pinecone

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()


### 3. Instantiate the Pinecone client

In [ ]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)


### 4. Define your query & documents

Five documents mixing "Apple" (the company) and "apple" (the fruit), so the reranker's
contextual understanding actually gets tested rather than trivially matching on the
word alone.

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "An apple a day keeps the doctor away, thanks to its fiber and vitamin C content.",  # apple fruit
    "Apple's latest iPhone features a faster chip and an improved camera system.",  # Apple company products
    "Red and green apples differ in sweetness, with Granny Smith being notably tart.",  # apple fruit
    "The MacBook Pro and iPad are among Apple's most popular product lines.",  # Apple company products
    "Tim Cook has led Apple as CEO since 2011, succeeding Steve Jobs.",  # Apple company, general
]


### 5. Call the reranker

In [ ]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,
)


### 6. Inspect reranked results

**Confirmed via the installed SDK's source** (`pinecone/data/features/inference/models/rerank_result.py`
wraps `pinecone/core/openapi/inference/model/rerank_result.py`, whose `openapi_types` are
exactly `model`, `data`, `usage`): the results list lives on `reranked.data`, not
`.matches`. Each item is a `RankedDocument` with `.score` and `.document` (confirmed via
`ranked_document.py`), and `.document` is a free-form object exposing whatever keys you
passed in -- here, `.text` (confirmed via `document.py`, which has no fixed fields at
all: it accepts arbitrary properties).

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1}. score={m.score:.4f} | {m.document.text}")

show_reranked_results(query, reranked.data)


## Part 2: Setup a Serverless Index for Medical Notes

### 1. Install data & model libraries

In [ ]:
!pip install pandas torch transformers


### 2. Import modules & define environment settings

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'


### 3. Create or recreate the index

`dimension=384` because that's what `all-MiniLM-L6-v2` (used in Part 5's embedding
function, and already baked into the sample data's `values` column) actually outputs --
confirmed directly against both the real model and the real dataset below, not just
assumed from the model's name. `metric='cosine'` as recommended for text embeddings.

In [ ]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,  # matches all-MiniLM-L6-v2's output size
    metric='cosine',
    spec=spec,
)


## Part 3: Load the Sample Data

### 1. Download & read JSONL

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/refs/heads/main/Week%208/W8D2/data.json"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)


**Confirmed by actually downloading and parsing this file:** 100 rows, exactly the
3 columns `id`, `values`, `metadata` that `upsert_from_dataframe` needs (it converts each
row straight into a vector record via `df.to_dict(orient="records")` -- confirmed by
reading `pinecone/data/index.py`, so column names have to match `id`/`values`/`metadata`
exactly, not just conceptually). Each `values` list has exactly 384 floats, matching the
index's dimension above. `metadata` is a small dict per row, e.g.
`{'symptoms': 'chest pain', 'tests': 'EKG, stress test'}` -- used later in Part 6.

### 2. Preview the DataFrame

In [ ]:
# Show head of the DataFrame
print("Data shape:", df.shape)  # (rows, columns)
df.head()


## Part 4: Upsert Data into the Index

### 1. Instantiate index client & upsert

In [ ]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)


### 2. Wait for availability

`> 0` -- we just want confirmation that *some* vectors have landed and are queryable, not
all 100 exactly (serverless indexing is eventually consistent, so waiting for an exact
count can spin forever if even one vector lags).

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()


## Part 5: Query & Embedding Function

### 1. Define your embedding function

`mean(dim=0)`, confirmed by actually running this exact model: `model_output.last_hidden_state`
comes out shape `(1, seq_len, 384)` (batch size 1); after `[0]` it's `(seq_len, 384)`, so
sequence length is dimension **0** and averaging over it collapses to a single `(384,)`
vector per input. `mean(dim=1)` would instead average over the embedding dimension
itself, producing a useless `(seq_len,)`-shaped output.

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding


### 2. Run a semantic search query

`question` is deliberately "patient has chest pain" -- confirmed against the real
downloaded dataset that a `chest pain` symptom row actually exists (id `P001`, with
`tests: EKG, stress test`), so this query has a genuinely relevant match to find rather
than an arbitrary guess. `top_k=5` is in the exercise's own suggested 5-10 range.

**A real bug in the challenge's own given code, found by testing it against the actual
Pinecone request model (no network call needed -- type validation happens locally before
any request is sent):** the skeleton has `index.query(vector=[query], ...)`. But `query`
here is already `get_embedding(...).tolist()` -- a flat list of 384 floats. Wrapping it
in `[query]` turns it into a list *containing one list*, and `Index.query()`'s `vector`
parameter is typed `List[float]`, not `List[List[float]]`. Constructing the request with
the nested form throws exactly this, confirmed directly:
```
PineconeApiTypeError: Invalid type for variable '0'. Required value type is float and
passed type was list at ['vector'][0]
```
Fixed below by passing `query` directly, unwrapped.

In [ ]:
# Build a query to search
question = "patient has chest pain"
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=query, top_k=5, include_metadata=True)  # `query` is already a flat vector -- no extra [ ]

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)


## Part 6: Display & Rerank Clinical Notes

### 1. Display initial search results

`match["score"]` and `match["metadata"]` -- confirmed via the installed SDK: query match
objects (`ScoredVector`) subclass the same OpenAPI base model as everything else in this
notebook, which implements `__getitem__`, so dict-style access works for every field on
the object, the same way `match["id"]` already does in the line right above these two.

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]}')
        print(f'      Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)


### 2. Prepare documents for reranking

`match['metadata']` -- the same field just printed above, now flattened into a single
searchable string per note.

In [ ]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]


### 3. Execute serverless reranking

`refined_query` narrows from the general "chest pain" search down to the specific
diagnostic workup mentioned in that same record's metadata (`EKG, stress test`) --
confirmed present in the real data, so this is testing the reranker's ability to prefer
the more specifically-matching note over other chest-pain-adjacent ones, not a made-up
example. `top_n=3`, in the exercise's own suggested 2-3 range.

In [ ]:
# Define a more specific query for reranking
refined_query = "chest pain that requires an EKG and stress test"

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)


### 4. Show reranked results

Same object shape as Part 1's reranker call: `.data` for the results list, `.score` on
each `RankedDocument`, and `.document.reranking_field` since `reranking_field` is the key
we ourselves put into each `transformed_documents` entry above -- `Document` (confirmed
in Part 1) exposes whatever keys it was given, there's no fixed `.text`-only field.

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score}')
        print(f'      Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)


### 5. Clean up (optional)

Run this when you're done to avoid unnecessary charges -- serverless indexes cost money
while they contain data.

In [ ]:
# Delete the index to save resources
pc.delete_index(name=index_name)


## 🎯 Success Criteria recap

- ✅ Authenticate with Pinecone -- Part 1, steps 2-3.
- ✅ Run basic document reranking with sensible results -- Part 1, steps 4-6 (fruit vs.
  company "Apple" documents, reranked by `bge-reranker-v2-m3`).
- ✅ Create and populate a serverless index with medical notes -- Parts 2-4.
- ✅ Execute semantic search queries on medical data -- Part 5 (with the `vector=[query]`
  bug fixed, since that call would otherwise fail before ever reaching the network).
- ✅ Compare original search results with reranked results -- Part 6, steps 1 and 4.
- ✅ Understand how reranking improves search relevance -- Part 6, step 3's
  `refined_query` narrows the same candidate set toward one specific note.